In [1]:
!pip install selenium pyarrow lxml requests beautifulsoup4 pandas
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
import glob
import pickle
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
}

YEAR = 2021
SAVE_DIR = rf'C:\\Users\\lcsse\\Desktop\\STAGE_GENT\\scrapping\\data\\pcs{YEAR}'
os.makedirs(SAVE_DIR, exist_ok=True)

dates = []
list_of_result_dfs = []
races = []
data_by_month = {}


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def is_page_not_found(response_text):
    return "Page not found" in response_text

def process_results_page(response_text, url):
    soup = BeautifulSoup(response_text, 'lxml')

    # Date
    try:
        date_tag = soup.select_one('div.value')
        date = date_tag.get_text(strip=True)
    except Exception:
        date = None
    dates.append(date)

    # Classification (2.UWT, 1.UWT, etc.)
    try:
        classif = None
        for li in soup.find_all('li'):
            if 'Classification:' in li.get_text():
                classif = li.get_text(strip=True).replace('Classification:', '').strip()
                break
    except Exception:
        classif = None

    # Startlist quality score
    try:
        startlist_quality = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Startlist quality score:' in text:
                match = re.search(r'Startlist quality score:\s*(\d+)', text)
                if match:
                    startlist_quality = int(match.group(1))
                break
    except Exception:
        startlist_quality = None

    # Average temperature
    try:
        avg_temperature = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Avg. temperature:' in text:
                match = re.search(r'Avg\. temperature:\s*([\d.]+)', text)
                if match:
                    avg_temperature = float(match.group(1))
                break
    except Exception:
        avg_temperature = None

    # Won how
    try:
        won_how = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Won how:' in text:
                won_how = text.split('Won how:')[-1].strip()
                break
    except Exception:
        won_how = None

    tables = soup.find_all('table')
    if tables:
        dfs = pd.read_html(str(tables))
        if dfs:
            df = dfs[0]
            df['date'] = date
            df['classification'] = classif
            df['startlist_quality'] = startlist_quality
            df['avg_temperature'] = avg_temperature
            df['won_how'] = won_how
            list_of_result_dfs.append(df)
            races.append(url)

def scrape_base_url(base_url):
    def has_results_table(response_text):
        soup = BeautifulSoup(response_text, 'lxml')
        return len(soup.find_all('table')) > 0

    result_url = base_url + "/result"
    response = requests.get(result_url, headers=headers)
    time.sleep(1)
    if has_results_table(response.text):
        process_results_page(response.text, result_url)

    stage = 1
    while True:
        stage_url = f"{base_url}/stage-{stage}"
        response = requests.get(stage_url, headers=headers)
        time.sleep(1)
        if has_results_table(response.text):
            process_results_page(response.text, stage_url)
            stage += 1
        else:
            break

    print(f"✅ {base_url}")

def scrape_urls(url_list):
    for base_url in url_list:
        scrape_base_url(base_url)

def clean_race_url(url):
    url = url.replace('/result', '')
    url = re.sub(r'/stage-\d+', '', url)
    return url

def save_month(month):
    for race, df, date in zip(data_by_month[month]['races'], data_by_month[month]['dfs'], data_by_month[month]['dates']):
        try:
            df['race_url'] = race
            df['date'] = date

            if '/stage-' in race:
                race_type = 'stage_' + race.split('/stage-')[-1]
            elif '/result' in race:
                race_type = 'result'
            else:
                race_type = 'one_day'

            race_name = race.replace('https://www.procyclingstats.com/race/', '')
            race_name = re.sub(r'/stage-\d+', '', race_name)
            race_name = race_name.replace('/result', '').replace('/', '_')

            clean_date = date.replace(' ', '_') if date else 'no_date'

            if 'classification' in df.columns and df['classification'].iloc[0]:
                classif = str(df['classification'].iloc[0]).strip()
            else:
                classif = 'no_classif'

            filename = f"{clean_date}__{race_type}__{race_name}__{classif}.csv"
            df.to_csv(os.path.join(SAVE_DIR, filename), index=False)
        except Exception as e:
            print(f"❌ {race}: {e}")
    print(f"✅ Mois {month}: {len(data_by_month[month]['races'])} fichiers sauvegardés")

In [3]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=1&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour janvier 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[1] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(1)
driver.quit()

✅ 13 URLs trouvées pour janvier 2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/new-zealand-cycle-classic/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/vuelta-al-tachira/2021


...

✅ https://www.procyclingstats.com/race/tour-de-langkawi/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-d-ouverture/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/great-ocean-road-race/2021
Total brut: 41 | Uniques: 12 | Manquantes: 1
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2021/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2021/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2021/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2021/stage-6: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2021/stage-7: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/r

In [4]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=2&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour fevrier 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[2] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(2)
driver.quit()

✅ 22 URLs trouvées pour fevrier 2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/etoile-de-besseges/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/herald-sun-tour/2021


...

✅ https://www.procyclingstats.com/race/la-drome-classic/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/kuurne-brussel-kuurne/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-uruguay/2021
Total brut: 45 | Uniques: 22 | Manquantes: 0
❌ https://www.procyclingstats.com/race/herald-sun-tour/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/herald-sun-tour/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/herald-sun-tour/2021/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/herald-sun-tour/2021/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/herald-sun-tour/2021/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-del-porvenir/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-del-porvenir/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-del-porvenir/2021/stage-3: single positional indexer is out-of

In [5]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=3&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour mars 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[3] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(3)
driver.quit()

✅ 39 URLs trouvées pour mars 2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-samyn/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/african-cycling-championships-ttt/2021


...

✅ https://www.procyclingstats.com/race/cholet-pays-de-loire/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-adria-mobil/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/dwars-door-vlaanderen/2021
Total brut: 67 | Uniques: 38 | Manquantes: 1
❌ https://www.procyclingstats.com/race/gp-de-la-ville-de-lillers/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/raport-tour/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/raport-tour/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/raport-tour/2021/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/raport-tour/2021/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/raport-tour/2021/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/olympias-tour/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/olympias-tour/2021/stage-2: single positional indexer is out-of-bounds
❌ https:

In [6]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=4&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour avril 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[4] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(4)
driver.quit()

✅ 27 URLs trouvées pour avril 2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/tour-of-mevlana/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gran-premio-miguel-indurain/2021


...

✅ https://www.procyclingstats.com/race/tour-de-romandie/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/tour-de-yorkshire/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/vuelta-asturias/2021
Total brut: 79 | Uniques: 27 | Manquantes: 0
❌ https://www.procyclingstats.com/race/grosser-osterpreis-bad-neuenahr-ahrweiler/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/region-pays-de-la-loire/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/region-pays-de-la-loire/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/region-pays-de-la-loire/2021/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/region-pays-de-la-loire/2021/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/klasika-primavera/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-loir-et-cher/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-l

In [7]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=5&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour mai 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[5] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(5)
driver.quit()

✅ 43 URLs trouvées pour mai 2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/memorial-andrzeja-trochanowskiego/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/ronde-van-overijssel/2021


...

✅ https://www.procyclingstats.com/race/albani-classic-fyen-rundt/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-slovenia/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/tour-auvergne-rhone-alpes/2021
Total brut: 124 | Uniques: 42 | Manquantes: 1
❌ https://www.procyclingstats.com/race/memorial-andrzeja-trochanowskiego/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-van-overijssel/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-international-a-la-cominudad-de-madrid/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-international-a-la-cominudad-de-madrid/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-international-a-la-cominudad-de-madrid/2021/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-international-a-la-cominudad-de-madrid/2021/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/fleche-ardennaise/2021/result: s

In [8]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=6&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour juin 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[6] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(6)
driver.quit()

✅ 151 URLs trouvées pour juin 2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofeo-alcide-degasperi/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/ronde-de-l-oise/2021


...

✅ https://www.procyclingstats.com/race/nc-iran-itt/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/int-wielertrofee-jong-maar-moedig-iwt/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/tour-of-bulgaria/2021
Total brut: 229 | Uniques: 149 | Manquantes: 2
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2021/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2021/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gp-de-la-ville-d-alger/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/sri-lanka-t-cup/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/sri-lanka-t-cup/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/sri-lanka-t-cup/2021/stage-3: single positional indexer is out-

In [9]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=7&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour juillet 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[7] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(7)
driver.quit()

✅ 47 URLs trouvées pour juillet 2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-algeria-itt/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/sibiu-cycling-tour/2021


...

✅ https://www.procyclingstats.com/race/scandinavian-race-uppsala/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/heist-op-den-berg/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/san-sebastian/2021
Total brut: 79 | Uniques: 46 | Manquantes: 1
❌ https://www.procyclingstats.com/race/2-districtenpijl-ekeren-deurne-me/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/trofeo-citta-di-brescia/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/chrono-kristin-armstrong/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-delta/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/volta-nxt-classic/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gp-cerami/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/scandinavian-race-uppsala/2021/result: single positional indexer is out-of-bounds
✅ Mois 7: 79 fichiers sauvegardés


In [10]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=8&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour aout 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[8] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(8)
driver.quit()

✅ 54 URLs trouvées pour aout 2021
✅ https://www.procyclingstats.com/race/tour-of-austria/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/circuito-de-getxo/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/vuelta-a-burgos/2021


...

✅ https://www.procyclingstats.com/race/ronde-van-de-achterhoek/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/renewi-tour/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/turul-romaniei/2021
Total brut: 145 | Uniques: 53 | Manquantes: 1
❌ https://www.procyclingstats.com/race/minsk-cup/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/grand-prix-minsk/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/puchar-uzdrowisk-karpackich/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/subway-3-stage-race/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/subway-3-stage-race/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/schaal-schels/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/cyclassics-hamburg/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/speedy-tour-d-indonesia/2021/stage-1: single positional index

In [11]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=9&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour septembre 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[9] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(9)
driver.quit()

✅ 63 URLs trouvées pour septembre 2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-uzbekistan-itt/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/elite-road-central-american-championships-itt/2021


...

✅ https://www.procyclingstats.com/race/tour-of-croatia/2021
✅ https://www.procyclingstats.com/race/tour-of-the-gila/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/circuit-franco-belge/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-ireland-itt/2021
Total brut: 115 | Uniques: 61 | Manquantes: 2
❌ https://www.procyclingstats.com/race/hafjell-gp/2021/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-xingtai/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-xingtai/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-xingtai/2021/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/dookola-warmii-i-mazur/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/dookola-warmii-i-mazur/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-taiwan/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-taiwan/2021/stage-2: single positional indexer is out-of-

In [12]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=10&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour octobre 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[10] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(10)
driver.quit()

✅ 59 URLs trouvées pour octobre 2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/route-adelie-de-vitre/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/tour-of-montenegro/2021


...

✅ https://www.procyclingstats.com/race/nc-indonesia/2021
✅ https://www.procyclingstats.com/race/tour-de-kumano/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/tour-du-faso/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/vuelta-ciclista-a-venezuela/2021
Total brut: 123 | Uniques: 58 | Manquantes: 1
❌ https://www.procyclingstats.com/race/tour-of-montenegro/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-montenegro/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-montenegro/2021/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-fuzhou/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-fuzhou/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-fuzhou/2021/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-fuzhou/2021/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-fuzhou/2021/stage-5: single positional indexer

In [13]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=11&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour novembre 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[11] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(11)
driver.quit()

✅ 3 URLs trouvées pour novembre 2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/tour-de-limpopo/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-india-itt/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-india-rr/2021
Total brut: 5 | Uniques: 3 | Manquantes: 0
❌ https://www.procyclingstats.com/race/tour-de-limpopo/2021/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-limpopo/2021/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-limpopo/2021/stage-3: single positional indexer is out-of-bounds
✅ Mois 11: 5 fichiers sauvegardés


In [14]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2021&month=12&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour decembre 2021")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[12] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(12)
driver.quit()

✅ 4 URLs trouvées pour decembre 2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/tour-of-thailand/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated an

✅ https://www.procyclingstats.com/race/vuelta-a-ecuador/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-malaysia-itt/2021


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_4464/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-malaysia/2021
Total brut: 16 | Uniques: 4 | Manquantes: 0
✅ Mois 12: 16 fichiers sauvegardés
